# Custom optimization examples

Use this notebook to define your own data and run the solvers from `nesterov_smoothing.py`.

It covers matrix games, continuous location, Maximum of absolute values, and Sum of absolute values.

In [16]:
import numpy as np

from nesterov_smoothing import (
    ContinuationConfig,
    solve_continuous_location,
    solve_paper_matrix_game,
    solve_piecewise_linear_max_abs,
    solve_sum_absolute_values
)

from nesterov_continuous_location import _sample_points_in_l2_ball


np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

def print_result(name, result):
    print(name)
    print("iterations:", result.iterations)
    print("predicted :", result.predicted_iterations)
    print("mu        :", result.mu)
    print("gap       :", result.gap)
    if hasattr(result, "objective_value"):
        print("value     :", result.objective_value)
    else:
        print("value     :", result.primal_value)
    if hasattr(result, "dual_value"):
        print("dual      :", result.dual_value)
    if hasattr(result, "theoretical_gap_bound"):
        print("bound     :", result.theoretical_gap_bound)
    print("x         :", result.x)
    if hasattr(result, "u"):
        print("u         :", result.u)
    print()


## 1. Custom matrix game

Define a payoff matrix `A` and solve the game.

In [19]:
A = np.array([
    [1.0, -1.0, 0.5],
    [-0.5, 2.0, -1.5],
    [0.2, -0.1, 1.0],
], dtype=np.float64)

# A = rng.uniform(-1.0, 1.0, size=(300, 1000))

matrix_result = solve_paper_matrix_game(
    A,
    epsilon=1e-3,
    check_frequency=1,
)

print_result("Matrix game", matrix_result)

Matrix game
iterations: 3508
predicted : 8789
mu        : 0.0004551196133134187
gap       : 0.0009997719166257024
value     : 0.22029767462875138
dual      : 0.21929790271212568
x         : [0.4923 0.3508 0.1569]
u         : [0.2521 0.2606 0.4873]



## 2. Custom continuous location problem

`cities` stores the points and `weights` stores their importance.

In [23]:
cities = np.array([
    [0.0, 0.0],
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [0.4, 0.7],
], dtype=np.float64)

weights = np.array([1.0, 2.0, 1.5, 1.0, 0.8], dtype=np.float64)


# cities = _sample_points_in_l2_ball(rng, count=50, dimension=5, radius=0.8)
# weights = rng.uniform(0.5, 1.5, size=50)


location_result = solve_continuous_location(
    cities=cities,
    weights=weights,
    radius=1.0,
    epsilon=1e-3,
    check_frequency=1,
)

print_result("Continuous location", location_result)

Continuous location
iterations: 8510
predicted : 12600
mu        : 0.00015873015873015873
gap       : 0.0009999851079314226
value     : 4.0501507708639375
dual      : 4.049150785756006
bound     : 0.0015959757477052232
x         : [0.4601 0.5976]



## 3. Custom Maximum of absolute values problem

This solves $\min_{\|x\| \le r} \max_j |a_j^T x - b_j|$.

In [41]:
A_pw = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
    [1.0, -1.0],
], dtype=np.float64)

b_pw = np.array([0.2, -0.1, 0.7, 0.0], dtype=np.float64)

# A_pw = rng.normal(size=(300, 200))
# b_pw = rng.normal(size=300)

piecewise_result = solve_piecewise_linear_max_abs(
    A=A_pw,
    b=b_pw,
    radius=1.0,
    epsilon=1e-3,
    check_frequency=1,
)

print_result("Maximum of absolute values", piecewise_result)

Maximum of absolute values
iterations: 2911
predicted : 5769
mu        : 0.00024044917348149393
gap       : 0.0009999810601405423
value     : 0.22508333333333333
dual      : 0.2240833522731928
x         : [0.35   0.1251]
u         : [0.     0.4998 0.     0.2494 0.     0.     0.2508 0.    ]



## 4. Custom Sum of absolute values problem

This solves $\min_{\|x\| \le r} \sum_j |a_j^T x - b_j|$.

In [28]:
A_sum = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, -1.0],
    [2.0, 1.0],
], dtype=np.float64)

b_sum = np.array([0.5, -0.2, 0.1, 1.0], dtype=np.float64)


# A_sum = rng.normal(size=(300, 200))
# b_sum = rng.normal(size=300)

sum_result = solve_sum_absolute_values(
    A=A_sum,
    b=b_sum,
    radius=1.0,
    epsilon=1e-3,
    check_frequency=1,
)

print_result("Sum of absolute values", sum_result)

Sum of absolute values
iterations: 5208
predicted : 10043
mu        : 0.00025
gap       : 0.000999724025482318
value     : 0.6
dual      : 0.5990002759745177
x         : [0.4261 0.1477]
u         : [-0.9996  1.      0.9997 -0.0008]



## 5. Optional: continuation

To test changing `\mu`, pass a `ContinuationConfig`.

In [11]:
continuation = ContinuationConfig(
    start_factor=4.0,
    decay=0.5,
    stage_factor=1.0,
)

matrix_continuation_result = solve_paper_matrix_game(
    A,
    epsilon=1e-3,
    check_frequency=1,
    continuation=continuation
)

print_result("Matrix game with continuation", matrix_continuation_result)

[(stage.index, stage.mu, stage.iterations, stage.target_met, stage.final_stage)
 for stage in matrix_continuation_result.continuation_stages]

Matrix game with continuation
iterations: 4090
predicted : 8789
mu        : 0.0004551196133134187
gap       : 0.0009996305778268155
value     : 0.2204460955144094
dual      : 0.2194464649365826
x         : [0.4921 0.3508 0.1571]
u         : [0.2524 0.2607 0.4869]



[(1, 0.0018204784532536748, 1362, True, False),
 (2, 0.0009102392266268374, 2728, True, False)]